# ClipForge — قالب تلقائي بنقرة واحدة

شغّل الخلية الوحيدة بالأسفل. ستفتح نافذة واحدة: اختر الفيديو، ويمكنك اختيار اللوجو معه اختياريًا. بعد الرفع سيطبق القالب كل التعديلات تلقائيًا ثم يبدأ تنزيل MP4. لا تكتب أي أوامر.

القالب ينتج فيديو عمودي 9:16 مناسب للريلز وتيك توك وشورتس، مع تحسين ألوان، عنوان، وقص تلقائي من منتصف الفيديو.


In [ ]:
#@title ClipForge: ارفع الفيديو وشغّل التصدير تلقائيًا
!apt-get -qq update && apt-get -qq install -y ffmpeg
from google.colab import files
from pathlib import Path
import subprocess, os

print("اختر الفيديو، ويمكنك اختيار Logo معه اختياريًا")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("لم يتم اختيار أي ملف")

video_ext = {".mp4", ".mov", ".webm", ".mkv", ".avi", ".m4v"}
image_ext = {".png", ".jpg", ".jpeg", ".webp"}
video = next((n for n in uploaded if Path(n).suffix.lower() in video_ext), None)
logo = next((n for n in uploaded if Path(n).suffix.lower() in image_ext), None)
if not video:
    raise RuntimeError("اختر ملف فيديو بصيغة MP4 أو MOV أو WebM")

# إعدادات القالب الثابت — لا تحتاج إلى تعديلها
TITLE = "ClipForge"
LOGO_POSITION = "top-right"
LOGO_WIDTH = 220
START_SECONDS = 0
END_SECONDS = 0
OUTPUT = "clipforge_ready.mp4"

filters = [
    "scale=1080:1920:force_original_aspect_ratio=increase",
    "crop=1080:1920",
    "eq=brightness=0.02:contrast=1.05:saturation=1.08",
    "setsar=1"
]
safe_title = TITLE.replace("\\", "").replace(":", "\\:").replace("'", "\\'")
if safe_title:
    filters.append(f"drawtext=text='{safe_title}':fontcolor=white:fontsize=64:borderw=4:bordercolor=black:x=(w-text_w)/2:y=h-text_h-100")
vf = ",".join(filters)
cmd = ["ffmpeg", "-y"]
if START_SECONDS > 0:
    cmd += ["-ss", str(START_SECONDS)]
cmd += ["-i", video]
if END_SECONDS > START_SECONDS:
    cmd += ["-t", str(END_SECONDS - START_SECONDS)]
if logo:
    positions = {"top-right":"W-w-36:36", "top-left":"36:36", "bottom-right":"W-w-36:H-h-36", "bottom-left":"36:H-h-36"}
    position = positions.get(LOGO_POSITION, positions["top-right"])
    complex_filter = f"[0:v]{vf}[base];[1:v]scale={LOGO_WIDTH}:-1[lg];[base][lg]overlay={position}[vout]"
    cmd += ["-i", logo, "-filter_complex", complex_filter, "-map", "[vout]", "-map", "0:a?"]
else:
    cmd += ["-vf", vf, "-map", "0:v", "-map", "0:a?"]
cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "22", "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart", OUTPUT]
print("جاري تطبيق القالب والتصدير... انتظر حتى تظهر رسالة الاكتمال")
subprocess.run(cmd, check=True)
print("اكتمل التصدير. سيبدأ التنزيل الآن:", OUTPUT)
files.download(OUTPUT)
